In [12]:
#IMPORTING NECESSARY LIBRARIES

import re  # used for pattern matching (like finding MOQ, ISO, etc.)
import time # used to add delays (so we don't spam websites)
import random
from concurrent.futures import ThreadPoolExecutor, as_completed # for parallel scraping
from datetime import datetime # to store when data was scraped
from pathlib import Path # to handle file paths safely
from urllib.parse import quote_plus, urlparse, urljoin

import pandas as pd  # used to store final data in table format
import requests  # used to download website pages
from bs4 import BeautifulSoup # used to extract data from HTML pages


In [13]:
# -------------------------------
# BASIC SETTINGS
# -------------------------------

# Pretending we are a real browser (important to avoid blocking)
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0 Safari/537.36"
    ),
    "Accept-Language": "en-IN,en;q=0.9",
}

TARGET_ROWS = 100 # how many companies we want in final dataset
REQUEST_TIMEOUT = 4 # stop request if website takes too long
POLITE_SLEEP_SECONDS = (0.2, 0.6)  # random delay between requests
ENABLE_LIVE_SEARCH = True # turn ON/OFF real Google/Bing search scraping
SEED_SCRAPE_WORKERS = 12 # number of parallel threads for speed

# output file names
OUTPUT_CSV = "indian_manufacturers_100.csv"
OUTPUT_XLSX = "indian_manufacturers_100.xlsx"

In [14]:
#These are broad queries to find manufacturers in different industries
SEARCH_QUERIES = [
    "India private label cosmetics manufacturer MOQ packaging certification",
    "India nutraceutical contract manufacturer MOQ GMP FSSAI",
    "India food contract manufacturer private label MOQ packaging",
    "India apparel garment manufacturer MOQ export certification",
    "India electronics EMS contract manufacturer ISO capacity",
    "India packaging manufacturer custom packaging MOQ",
    "India ayurvedic herbal product private label manufacturer GMP",
    "India personal care private label manufacturer MOQ",
    "India home care cleaning products contract manufacturer",
    "India toy manufacturer custom OEM MOQ",
    "India stationery manufacturer custom branding MOQ",
    "India leather goods manufacturer private label MOQ",
    "India footwear manufacturer private label MOQ",
    "India jewellery manufacturer custom OEM MOQ",
    "India plastic injection molding manufacturer ISO MOQ",
    "India metal fabrication contract manufacturer ISO capacity",
    "India medical device contract manufacturer ISO 13485",
    "India pharmaceutical CDMO manufacturer GMP capacity",
    "India textile manufacturer private label MOQ",
    "India furniture manufacturer custom OEM MOQ",
]

# -------------------------------
# MANUAL COMPANY LIST (backup data)
# -------------------------------
# These are trusted known companies used if scraping fails

SEED_MANUFACTURERS = [
    ("Dixon Technologies", "https://www.dixoninfo.com/", "Noida, Uttar Pradesh, India", "Electronics / EMS"),
    ("Syrma SGS Technology", "https://www.syrmasgs.com/", "Chennai, Tamil Nadu, India", "Electronics / EMS"),
    ("Kaynes Technology", "https://www.kaynestechnology.co.in/", "Mysuru, Karnataka, India", "Electronics / EMS"),
    ("Avalon Technologies", "https://www.avalontec.com/", "Chennai, Tamil Nadu, India", "Electronics / EMS"),
    ("Centum Electronics", "https://www.centumelectronics.com/", "Bengaluru, Karnataka, India", "Electronics / EMS"),
    ("VVDN Technologies", "https://www.vvdntech.com/", "Gurugram, Haryana, India", "Electronics / EMS; OEM/ODM"),
    ("PG Electroplast", "https://www.pgel.in/", "Greater Noida, Uttar Pradesh, India", "Electronics / appliances manufacturing"),
    ("Elin Electronics", "https://www.elinindia.com/", "New Delhi, Delhi, India", "Electronics / EMS"),
    ("Amber Enterprises", "https://www.ambergroupindia.com/", "Gurugram, Haryana, India", "Consumer durables / HVAC manufacturing"),
    ("SFO Technologies", "https://www.sfo.co.in/", "Kochi, Kerala, India", "Electronics / EMS"),
    ("Akums Drugs & Pharmaceuticals", "https://www.akums.in/", "Haridwar, Uttarakhand, India", "Pharmaceuticals / CDMO"),
    ("Windlas Biotech", "https://www.windlas.com/", "Dehradun, Uttarakhand, India", "Pharmaceuticals / CDMO"),
    ("Tirupati Group", "https://www.tirupatigroup.co.in/", "Paonta Sahib, Himachal Pradesh, India", "Pharmaceuticals / nutraceuticals / CDMO"),
    ("Innova Captab", "https://www.innovacaptab.com/", "Panchkula, Haryana, India", "Pharmaceuticals / CDMO"),
    ("Piramal Pharma Solutions", "https://www.piramalpharmasolutions.com/", "Mumbai, Maharashtra, India", "Pharmaceuticals / CDMO"),
    ("Syngene International", "https://www.syngeneintl.com/", "Bengaluru, Karnataka, India", "Biologics / pharma research and manufacturing"),
    ("Sai Life Sciences", "https://www.sailife.com/", "Hyderabad, Telangana, India", "Pharmaceuticals / CDMO"),
    ("Aragen Life Sciences", "https://www.aragen.com/", "Hyderabad, Telangana, India", "Pharmaceuticals / CDMO"),
    ("Anthem Biosciences", "https://www.anthembio.com/", "Bengaluru, Karnataka, India", "Pharmaceuticals / CDMO"),
    ("Laurus Labs", "https://www.lauruslabs.com/", "Hyderabad, Telangana, India", "Pharmaceuticals / ingredients"),
    ("Divi's Laboratories", "https://www.divislabs.com/", "Hyderabad, Telangana, India", "Pharmaceuticals / APIs"),
    ("Granules India", "https://www.granulesindia.com/", "Hyderabad, Telangana, India", "Pharmaceuticals / finished dosage / APIs"),
    ("Hetero", "https://www.heteroworld.com/", "Hyderabad, Telangana, India", "Pharmaceuticals"),
    ("Gland Pharma", "https://www.glandpharma.com/", "Hyderabad, Telangana, India", "Injectable pharmaceuticals"),
    ("Suven Pharmaceuticals", "https://www.suvenpharm.com/", "Hyderabad, Telangana, India", "Pharmaceuticals / CDMO"),
    ("HCP Wellness", "https://www.hcpwellness.in/", "Ahmedabad, Gujarat, India", "Cosmetics / personal care private label"),
    ("Vive Cosmetics", "https://www.vivecosmetic.com/", "Chandigarh, Chandigarh, India", "Cosmetics / personal care private label"),
    ("BO International", "https://www.bointernational.net/", "Noida, Uttar Pradesh, India", "Cosmetics / personal care private label"),
    ("Clarion Cosmetics", "https://www.clarioncosmetics.com/", "Chennai, Tamil Nadu, India", "Cosmetics / personal care manufacturing"),
    ("Vasa Cosmetics", "https://www.vasacosmetics.com/", "Mumbai, Maharashtra, India", "Cosmetics / personal care manufacturing"),
    ("Zoic Cosmetics", "https://www.zoiccosmetic.com/", "Mohali, Punjab, India", "Cosmetics / personal care private label"),
    ("Scot Derma", "https://www.scotderma.com/", "Chandigarh, Chandigarh, India", "Derma / cosmetics private label"),
    ("Lifevision Cosmetics", "https://www.lifevisioncosmetics.com/", "Chandigarh, Chandigarh, India", "Cosmetics / personal care private label"),
    ("AG Organica", "https://www.agorganica.com/", "Noida, Uttar Pradesh, India", "Essential oils / cosmetics ingredients"),
    ("Vanesa Care", "https://www.vanesacare.com/", "New Delhi, Delhi, India", "Personal care / aerosol manufacturing"),
    ("Zeon Lifesciences", "https://www.zeonlifesciences.com/", "Paonta Sahib, Himachal Pradesh, India", "Nutraceuticals / supplements"),
    ("Tirupati Wellness", "https://www.tirupatiwellness.com/", "Paonta Sahib, Himachal Pradesh, India", "Nutraceuticals / wellness products"),
    ("Herbal Hills", "https://www.herbalhills.in/", "Mumbai, Maharashtra, India", "Ayurveda / herbal products"),
    ("Dabur India", "https://www.dabur.com/", "Ghaziabad, Uttar Pradesh, India", "Ayurveda / personal care / food"),
    ("Himalaya Wellness", "https://himalayawellness.in/", "Bengaluru, Karnataka, India", "Ayurveda / personal care / wellness"),
    ("Baidyanath", "https://www.baidyanath.co.in/", "Kolkata, West Bengal, India", "Ayurveda / herbal products"),
    ("Patanjali Ayurved", "https://www.patanjaliayurved.org/", "Haridwar, Uttarakhand, India", "Ayurveda / food / personal care"),
    ("Emami", "https://www.emamiltd.in/", "Kolkata, West Bengal, India", "Personal care / healthcare products"),
    ("Charak Pharma", "https://www.charak.com/", "Mumbai, Maharashtra, India", "Ayurveda / herbal healthcare"),
    ("Kerala Ayurveda", "https://www.keralaayurveda.biz/", "Aluva, Kerala, India", "Ayurveda / herbal products"),
    ("UFlex", "https://www.uflexltd.com/", "Noida, Uttar Pradesh, India", "Flexible packaging"),
    ("Huhtamaki India", "https://www.huhtamaki.com/en-in/", "Mumbai, Maharashtra, India", "Flexible packaging"),
    ("Parksons Packaging", "https://www.parksonspackaging.com/", "Mumbai, Maharashtra, India", "Folding cartons / paper packaging"),
    ("TCPL Packaging", "https://www.tcpl.in/", "Mumbai, Maharashtra, India", "Paperboard packaging / labels"),
    ("EPL", "https://www.eplglobal.com/", "Mumbai, Maharashtra, India", "Laminated tubes / packaging"),
    ("Mold-Tek Packaging", "https://www.moldtekpackaging.com/", "Hyderabad, Telangana, India", "Rigid plastic packaging"),
    ("Manjushree Technopack", "https://www.manjushreeindia.com/", "Bengaluru, Karnataka, India", "Rigid plastic packaging"),
    ("AGI Greenpac", "https://www.agigreenpac.com/", "Gurugram, Haryana, India", "Glass packaging"),
    ("Pragati Pack", "https://www.pragati.com/", "Hyderabad, Telangana, India", "Printed packaging"),
    ("Cosmo First", "https://www.cosmofirst.com/", "New Delhi, Delhi, India", "Films / packaging"),
    ("Jindal Poly Films", "https://www.jindalpoly.com/", "New Delhi, Delhi, India", "Packaging films"),
    ("Flexituff Ventures", "https://www.flexituff.com/", "Pithampur, Madhya Pradesh, India", "Technical textiles / packaging"),
    ("Shahi Exports", "https://www.shahi.co.in/", "Bengaluru, Karnataka, India", "Apparel / textiles"),
    ("Gokaldas Exports", "https://www.gokaldasexports.com/", "Bengaluru, Karnataka, India", "Apparel manufacturing"),
    ("Pearl Global", "https://www.pearlglobal.com/", "Gurugram, Haryana, India", "Apparel manufacturing"),
    ("Arvind", "https://www.arvind.com/", "Ahmedabad, Gujarat, India", "Textiles / apparel"),
    ("KPR Mill", "https://www.kprmilllimited.com/", "Coimbatore, Tamil Nadu, India", "Textiles / apparel"),
    ("Kitex Garments", "https://www.kitexgarments.com/", "Kizhakkambalam, Kerala, India", "Garments / kidswear"),
    ("Eastman Exports", "https://www.eastmanexports.com/", "Tiruppur, Tamil Nadu, India", "Apparel manufacturing"),
    ("SP Apparels", "https://www.spapparels.com/", "Avinashi, Tamil Nadu, India", "Apparel manufacturing"),
    ("Loyal Textile Mills", "https://www.loyaltextiles.com/", "Chennai, Tamil Nadu, India", "Textiles / apparel"),
    ("Vardhman Textiles", "https://www.vardhman.com/", "Ludhiana, Punjab, India", "Textiles / yarn / fabric"),
    ("Trident Group", "https://www.tridentindia.com/", "Ludhiana, Punjab, India", "Home textiles / paper"),
    ("Nahar Group", "https://www.naharindia.com/", "Ludhiana, Punjab, India", "Textiles / apparel"),
    ("Raymond", "https://www.raymond.in/", "Mumbai, Maharashtra, India", "Textiles / apparel"),
    ("Welspun India", "https://www.welspunindia.com/", "Mumbai, Maharashtra, India", "Home textiles"),
    ("Himatsingka", "https://www.himatsingka.com/", "Bengaluru, Karnataka, India", "Home textiles"),
    ("ADF Foods", "https://www.adf-foods.com/", "Mumbai, Maharashtra, India", "Food manufacturing"),
    ("Mrs. Bectors Food Specialities", "https://www.mrsbectorfoods.com/", "Gurugram, Haryana, India", "Bakery / food manufacturing"),
    ("Bikaji Foods", "https://www.bikaji.com/", "Bikaner, Rajasthan, India", "Snacks / food manufacturing"),
    ("Haldiram's", "https://www.haldirams.com/", "Nagpur, Maharashtra, India", "Snacks / food manufacturing"),
    ("Prataap Snacks", "https://www.prataapsnacks.com/", "Indore, Madhya Pradesh, India", "Snacks / food manufacturing"),
    ("LT Foods", "https://www.ltgroup.in/", "Gurugram, Haryana, India", "Rice / food products"),
    ("KRBL", "https://www.krblrice.com/", "Noida, Uttar Pradesh, India", "Rice / food products"),
    ("Tasty Bite", "https://www.tastybite.co.in/", "Pune, Maharashtra, India", "Ready-to-eat food manufacturing"),
    ("Heritage Foods", "https://www.heritagefoods.in/", "Hyderabad, Telangana, India", "Dairy / food products"),
    ("Hatsun Agro Product", "https://www.hatsun.com/", "Chennai, Tamil Nadu, India", "Dairy / food products"),
    ("Supreme Industries", "https://www.supreme.co.in/", "Mumbai, Maharashtra, India", "Plastics / molded products"),
    ("Nilkamal", "https://www.nilkamal.com/", "Mumbai, Maharashtra, India", "Plastics / furniture"),
    ("Time Technoplast", "https://www.timetechnoplast.com/", "Mumbai, Maharashtra, India", "Industrial packaging / plastics"),
    ("Cello World", "https://www.celloworld.com/", "Mumbai, Maharashtra, India", "Consumer products / plastics"),
    ("Hamilton Housewares", "https://www.hamiltonindia.in/", "Mumbai, Maharashtra, India", "Housewares / plastics"),
    ("Godrej Interio", "https://www.godrejinterio.com/", "Mumbai, Maharashtra, India", "Furniture"),
    ("Featherlite", "https://www.featherlitefurniture.com/", "Bengaluru, Karnataka, India", "Furniture"),
    ("Wipro Furniture", "https://www.wipro.com/lighting-furniture/", "Bengaluru, Karnataka, India", "Furniture / workplace products"),
    ("Bharat Forge", "https://www.bharatforge.com/", "Pune, Maharashtra, India", "Metal forging / engineering"),
    ("JBM Group", "https://www.jbmgroup.com/", "Gurugram, Haryana, India", "Automotive / metal systems"),
    ("Poly Medicure", "https://www.polymedicure.com/", "Faridabad, Haryana, India", "Medical devices"),
    ("Meril Life Sciences", "https://www.merillife.com/", "Vapi, Gujarat, India", "Medical devices"),
    ("Hindustan Syringes & Medical Devices", "https://www.hmdhealthcare.com/", "Faridabad, Haryana, India", "Medical devices"),
    ("Romsons", "https://www.romsons.com/", "Agra, Uttar Pradesh, India", "Medical devices"),
    ("Healthium Medtech", "https://www.healthiummedtech.com/", "Bengaluru, Karnataka, India", "Medical devices"),
    ("BPL Medical Technologies", "https://www.bplmedicaltechnologies.com/", "Bengaluru, Karnataka, India", "Medical devices"),
    ("Trivitron Healthcare", "https://www.trivitron.com/", "Chennai, Tamil Nadu, India", "Medical devices"),
    ("Sahajanand Medical Technologies", "https://www.smtl.com/", "Surat, Gujarat, India", "Medical devices"),
]

# -------------------------------
# BLOCKED WEBSITES
# -------------------------------
# We don't want social media or shopping sites
BLOCKED_DOMAINS = {
    "google.com",
    "youtube.com",
    "facebook.com",
    "instagram.com",
    "linkedin.com",
    "pinterest.com",
    "twitter.com",
    "x.com",
    "wikipedia.org",
    "amazon.in",
    "flipkart.com",
    "justdial.com",
}

# -------------------------------
# KEYWORDS TO IDENTIFY MANUFACTURERS
# -------------------------------
# If page contains these words → likely a manufacturer
MANUFACTURER_KEYWORDS = [
    "manufacturer",
    "manufacturing",
    "contract manufacturing",
    "private label",
    "white label",
    "oem",
    "odm",
    "factory",
    "production",
    "cdmo",
    "ems",
]

CERTIFICATION_PATTERNS = [
    r"\bISO\s?9001\b",
    r"\bISO\s?14001\b",
    r"\bISO\s?13485\b",
    r"\bISO\s?22000\b",
    r"\bGMP\b",
    r"\bWHO-GMP\b",
    r"\bUSFDA\b",
    r"\bFDA\b",
    r"\bFSSAI\b",
    r"\bHACCP\b",
    r"\bBRC\b",
    r"\bSEDEX\b",
    r"\bSA8000\b",
    r"\bGOTS\b",
    r"\bOEKO[- ]?TEX\b",
    r"\bCE\b",
    r"\bRoHS\b",
    r"\bBIS\b",
    r"\bHalal\b",
    r"\bKosher\b",
]

CATEGORY_RULES = [
    ("cosmetics", "Cosmetics / personal care"),
    ("skin care", "Cosmetics / personal care"),
    ("personal care", "Cosmetics / personal care"),
    ("nutraceutical", "Nutraceuticals / supplements"),
    ("supplement", "Nutraceuticals / supplements"),
    ("food", "Food and beverages"),
    ("fssai", "Food and beverages"),
    ("garment", "Apparel / textiles"),
    ("apparel", "Apparel / textiles"),
    ("textile", "Apparel / textiles"),
    ("electronics", "Electronics / EMS"),
    ("ems", "Electronics / EMS"),
    ("packaging", "Packaging"),
    ("ayurvedic", "Ayurveda / herbal products"),
    ("herbal", "Ayurveda / herbal products"),
    ("cleaning", "Home care / cleaning products"),
    ("detergent", "Home care / cleaning products"),
    ("toy", "Toys"),
    ("stationery", "Stationery"),
    ("leather", "Leather goods"),
    ("footwear", "Footwear"),
    ("jewellery", "Jewellery"),
    ("jewelry", "Jewellery"),
    ("plastic", "Plastic products / molding"),
    ("injection molding", "Plastic products / molding"),
    ("metal", "Metal fabrication"),
    ("fabrication", "Metal fabrication"),
    ("medical device", "Medical devices"),
    ("pharmaceutical", "Pharmaceuticals / CDMO"),
    ("cdmo", "Pharmaceuticals / CDMO"),
    ("furniture", "Furniture"),
]

INDIAN_STATES = [
    "Andhra Pradesh", "Arunachal Pradesh", "Assam", "Bihar", "Chhattisgarh",
    "Delhi", "Goa", "Gujarat", "Haryana", "Himachal Pradesh", "Jharkhand",
    "Karnataka", "Kerala", "Madhya Pradesh", "Maharashtra", "Manipur",
    "Meghalaya", "Mizoram", "Nagaland", "Odisha", "Punjab", "Rajasthan",
    "Sikkim", "Tamil Nadu", "Telangana", "Tripura", "Uttar Pradesh",
    "Uttarakhand", "West Bengal",
]


In [15]:
# -------------------------------
# FUNCTIONS: BASIC HELPERS
# -------------------------------

# wait a small random time so websites don’t block us
def sleep_politely():
    time.sleep(random.uniform(*POLITE_SLEEP_SECONDS))

# remove extra spaces and clean messy text
def clean_text(text):
    text = re.sub(r"\s+", " ", text or "")
    return text.strip()

# extract website name from full URL
def domain_from_url(url):
    parsed = urlparse(url)
    domain = parsed.netloc.lower().replace("www.", "")
    return domain

# skip social media and unwanted websites
def is_blocked_domain(url):
    domain = domain_from_url(url)
    return any(domain == blocked or domain.endswith("." + blocked) for blocked in BLOCKED_DOMAINS)

# -------------------------------
# FETCH WEBSITE DATA
# -------------------------------


def safe_get(url):
    # safely open a website
    try:
        response = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
        if response.status_code >= 400:
            return None
        content_type = response.headers.get("content-type", "").lower()
        if "text/html" not in content_type and "application/xhtml" not in content_type:
            return None
        return response
    except requests.RequestException:
        return None

# -------------------------------
# SEARCH FUNCTIONS (Bing + DuckDuckGo)
# -------------------------------

def bing_search(query, max_results=12):
     # search on Bing and extract result links
    """Return organic result URLs from Bing HTML search."""
    search_url = f"https://www.bing.com/search?q={quote_plus(query)}&setmkt=en-IN"
    response = safe_get(search_url)
    if response is None:
        return []

    soup = BeautifulSoup(response.text, "html.parser")
    urls = []

    for result in soup.select("li.b_algo h2 a"):
        href = result.get("href")
        if not href or not href.startswith("http"):
            continue
        if is_blocked_domain(href):
            continue
        urls.append(href)

    # Fallback: search pages occasionally change their class names. This scans
    # normal links and keeps only likely company/result links.
    if not urls:
        for result in soup.select("a[href^='http']"):
            href = result.get("href")
            if not href or is_blocked_domain(href):
                continue
            if "bing.com" in domain_from_url(href):
                continue
            urls.append(href)

    # Keep order but remove duplicates.
    unique_urls = []
    seen = set()
    for url in urls:
        domain = domain_from_url(url)
        if domain not in seen:
            seen.add(domain)
            unique_urls.append(url)
        if len(unique_urls) >= max_results:
            break
    return unique_urls

def duckduckgo_search(query, max_results=12):
    # fallback search if Bing fails
    """Fallback search using DuckDuckGo's lightweight HTML endpoint."""
    search_url = f"https://duckduckgo.com/html/?q={quote_plus(query)}"
    response = safe_get(search_url)
    if response is None:
        return []

    soup = BeautifulSoup(response.text, "html.parser")
    urls = []
    for result in soup.select("a.result__a, a[href^='http']"):
        href = result.get("href")
        if not href or not href.startswith("http"):
            continue
        if is_blocked_domain(href):
            continue
        urls.append(href)

    unique_urls = []
    seen = set()
    for url in urls:
        domain = domain_from_url(url)
        if domain not in seen:
            seen.add(domain)
            unique_urls.append(url)
        if len(unique_urls) >= max_results:
            break
    return unique_urls


# -------------------------------
# SCRAPING PAGE CONTENT
# -------------------------------

def search_web(query, max_results=12):
    # open webpage and extract title + text
    # remove scripts, styles, etc.
    urls = bing_search(query, max_results=max_results)
    if urls:
        return urls
    return duckduckgo_search(query, max_results=max_results)


def extract_page_text_and_title(url):
    response = safe_get(url)
    if response is None:
        return "", "", url

    soup = BeautifulSoup(response.text, "html.parser")
    for tag in soup(["script", "style", "noscript", "svg"]):
        tag.decompose()

    title = clean_text(soup.title.get_text(" ")) if soup.title else ""
    paragraph_text = " ".join([p.get_text(" ", strip=True) for p in soup.find_all("p")])
    text = clean_text(paragraph_text)

    canonical = url
    canonical_tag = soup.find("link", rel=lambda value: value and "canonical" in value.lower())
    if canonical_tag and canonical_tag.get("href"):
        canonical = urljoin(url, canonical_tag["href"])

    return title, text[:120000], canonical


def extract_site_text_and_title(home_url):
    # scrape homepage + about + certifications pages
    # combine all text into one big dataset
    """Scrape the homepage plus a few common detail pages on the same domain."""
    parsed = urlparse(home_url)
    root = f"{parsed.scheme}://{parsed.netloc}"
    candidate_paths = [
        "",
        "/about-us",
        "/about",
        "/certifications",
    ]

    titles = []
    texts = []
    source_url = home_url

    for path in candidate_paths:
        url = root + path
        title, text, canonical = extract_page_text_and_title(url)
        if text:
            titles.append(title)
            texts.append(text)
            source_url = canonical or url
        if len(" ".join(texts)) > 180000:
            break

    combined_title = next((title for title in titles if title), "")
    combined_text = clean_text(" ".join(texts))[:220000]
    return combined_title, combined_text, source_url


# %%
def guess_name(title, url):
    domain = domain_from_url(url)
    if title:
        title = re.split(r"\s+[-|–]\s+|\s+\|\s+", title)[0]
        title = re.sub(r"\b(Home|Official Website|Manufacturer|Manufacturers)\b", "", title, flags=re.I)
        title = clean_text(title)
        if 2 <= len(title) <= 80:
            return title
    return domain.split(".")[0].replace("-", " ").title()


# -------------------------------
# EXTRACT COMPANY INFORMATION
# -------------------------------

def extract_location(text):
     # find Indian state/city from text
    ...
    found_state = None
    for state in INDIAN_STATES:
        if re.search(rf"\b{re.escape(state)}\b", text, flags=re.I):
            found_state = state
            break

    # Common city-before-state pattern, for example "Ahmedabad, Gujarat".
    if found_state:
        pattern = rf"([A-Z][A-Za-z .'-]{{2,40}}),\s*{re.escape(found_state)}"
        match = re.search(pattern, text)
        city = clean_text(match.group(1)) if match else "Not stated on source"
        return f"{city}, {found_state}, India"

    if re.search(r"\bIndia\b", text, flags=re.I):
        return "Not stated, India"
    return "Not stated on source"


def extract_categories(text):
    # detect industry category (food, pharma, etc.)
    ...
    lower_text = text.lower()
    categories = []
    for keyword, category in CATEGORY_RULES:
        if keyword in lower_text and category not in categories:
            categories.append(category)
    return "; ".join(categories[:4]) if categories else "Manufacturing / OEM"


def extract_moq(text):
     # find minimum order quantity info
    ...
    patterns = [
        r"\bMOQ\b.{0,80}",
        r"\bminimum order quantity\b.{0,100}",
        r"\bminimum order\b.{0,80}",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.I)
        if match:
            return clean_text(match.group(0))
    return "Not stated on source"


def extract_capacity(text):
    patterns = [
        r"\bproduction capacity\b.{0,120}",
        r"\bmanufacturing capacity\b.{0,120}",
        r"\bcapacity\b.{0,80}(?:units|pieces|pcs|tonnes|tons|kg|litres|liters|bottles|tablets|capsules)",
        r"\b\d+[,.]?\d*\s*(?:units|pieces|pcs|tonnes|tons|kg|litres|liters|bottles|tablets|capsules)\s*(?:per|/)\s*(?:day|month|year)",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.I)
        if match:
            return clean_text(match.group(0))
    return "Not stated on source"


def extract_certifications(text):
    found = set()
    for pattern in CERTIFICATION_PATTERNS:
        matches = re.findall(pattern, text, flags=re.I)
        for match in matches:
            found.add(match.upper().replace(" ", "").replace("-", ""))
    return "; ".join(list(found)[:6]) if found else "Not stated"


def extract_support(text, keywords, label):
    lower_text = text.lower()
    matched = [word for word in keywords if word in lower_text]
    if matched:
        return label
    return "Not stated on source"


def extract_lead_time(text):
    patterns = [
        r"\blead time\b.{0,100}",
        r"\bdelivery time\b.{0,100}",
        r"\b\d+\s*[-–to]+\s*\d+\s*(?:days|weeks)\b",
        r"\b\d+\s*(?:days|weeks)\s*(?:after|from)\s*(?:order|payment|approval)",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.I)
        if match:
            return clean_text(match.group(0))
    return "Not stated on source"


# -------------------------------
# MAIN LOGIC: CHECK IF COMPANY IS VALID
# -------------------------------
def looks_like_manufacturer(text):
    # check if page looks like manufacturing company
    lower_text = text.lower()
    return any(keyword in lower_text for keyword in MANUFACTURER_KEYWORDS)


# %%
def build_record(url):
     # 1. scrape website
    # 2. check if it's manufacturer
    # 3. extract all details
    # 4. return structured row (like Excel row)
    title, text, source_url = extract_site_text_and_title(url)
    if not text or "manufactur" not in text.lower() or not looks_like_manufacturer(text):
        return None

    logistics_support = extract_support(
        text,
        ["logistics", "shipping", "export", "freight", "dispatch", "delivery", "supply chain"],
        "Mentions shipping/export/logistics support",
    )
    packaging_support = extract_support(
        text,
        ["packaging", "private label", "custom label", "customized", "customization", "branding", "oem", "odm"],
        "Mentions packaging/customization/private-label support",
    )

    return {
        "name": guess_name(title, source_url),
        "website": f"{urlparse(source_url).scheme}://{urlparse(source_url).netloc}",
        "location (city, state, country)": extract_location(text),
        "categories served": extract_categories(text),
        "MOQ range": extract_moq(text),
        "production capacity": extract_capacity(text),
        "certifications": extract_certifications(text),
        "logistics support": logistics_support,
        "packaging/customization support": packaging_support,
        "lead time": extract_lead_time(text),
        "source link": source_url,
        "confidence_score": round(min(len(text) / 10000, 1), 2),
        "scraped_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }


def seed_record(name, website, location, categories):
    """Create a conservative record from the curated source list and website scrape."""
    scraped = build_record(website)
    if scraped:
        scraped["name"] = name
        scraped["website"] = website.rstrip("/")
        scraped["location (city, state, country)"] = location
        if scraped["categories served"] == "Manufacturing / OEM":
            scraped["categories served"] = categories
        else:
            scraped["categories served"] = categories
        return scraped

    support = "Not stated on source"
    if any(word in categories.lower() for word in ["private label", "oem", "odm", "packaging"]):
        packaging = "Relevant category; verify customization details on source"
    else:
        packaging = "Not stated on source"

    return {
        "name": name,
        "website": website.rstrip("/"),
        "location (city, state, country)": location,
        "categories served": categories,
        "MOQ range": "Not stated on source",
        "production capacity": "Not stated on source",
        "certifications": "Not stated on source",
        "logistics support": support,
        "packaging/customization support": packaging,
        "lead time": "Not stated on source",
        "source link": website,
        "confidence_score": 0,
        "scraped_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }




In [16]:
# -------------------------------
# RUN SCRAPER
# -------------------------------
records = []
seen_domains = set()
checked_domains = set()

if ENABLE_LIVE_SEARCH:
    for query in SEARCH_QUERIES:
        # search web for manufacturer websites
        print(f"Searching: {query}")
        result_urls = search_web(query, max_results=12)
        sleep_politely()

        for url in result_urls:
            domain = domain_from_url(url)
            if domain in checked_domains:
                continue
            checked_domains.add(domain)

            print(f"  Checking: {domain}")
            record = build_record(url)
            sleep_politely()

            if record:
                domain = domain_from_url(record["website"])
                if domain in seen_domains:
                    continue
                seen_domains.add(domain)
                records.append(record)
                print(f"    Added #{len(records)}: {record['name']}")

            if len(records) >= TARGET_ROWS:
                break

        if len(records) >= TARGET_ROWS:
            break
else:
    print("Live search disabled. Enriching curated 100-company manufacturer list directly.")

print(f"\nCollected {len(records)} manufacturer records.")

if len(records) < TARGET_ROWS:
    print("Filling remaining rows from curated Indian manufacturer seed list.")
    existing = {record["website"].replace("https://", "").replace("http://", "").replace("www.", "").rstrip("/") for record in records}
    pending_items = []
    for item in SEED_MANUFACTURERS:
        key = item[1].replace("https://", "").replace("http://", "").replace("www.", "").rstrip("/")
        if key not in existing:
            pending_items.append(item)

    with ThreadPoolExecutor(max_workers=SEED_SCRAPE_WORKERS) as executor:
        future_to_item = {executor.submit(seed_record, *item): item for item in pending_items}
        for future in as_completed(future_to_item):
            fallback = future.result()
            key = fallback["website"].replace("https://", "").replace("http://", "").replace("www.", "").rstrip("/")
            if key in existing:
                continue
            records.append(fallback)
            existing.add(key)
            print(f"  Added #{len(records)}: {fallback['name']}")
            if len(records) >= TARGET_ROWS:
                break

print(f"Final row count: {len(records)}")

Searching: India private label cosmetics manufacturer MOQ packaging certification
Searching: India nutraceutical contract manufacturer MOQ GMP FSSAI
Searching: India food contract manufacturer private label MOQ packaging
Searching: India apparel garment manufacturer MOQ export certification
Searching: India electronics EMS contract manufacturer ISO capacity
Searching: India packaging manufacturer custom packaging MOQ
Searching: India ayurvedic herbal product private label manufacturer GMP
Searching: India personal care private label manufacturer MOQ
Searching: India home care cleaning products contract manufacturer
Searching: India toy manufacturer custom OEM MOQ
Searching: India stationery manufacturer custom branding MOQ
Searching: India leather goods manufacturer private label MOQ
Searching: India footwear manufacturer private label MOQ
Searching: India jewellery manufacturer custom OEM MOQ
Searching: India plastic injection molding manufacturer ISO MOQ
Searching: India metal fabric

In [17]:
# -------------------------------
# FILL WITH BACKUP COMPANIES
# -------------------------------
# if scraping fails, use known companies list

# -------------------------------
# SAVE DATA
# -------------------------------

columns = [
    "name",
    "website",
    "location (city, state, country)",
    "categories served",
    "MOQ range",
    "production capacity",
    "certifications",
    "logistics support",
    "packaging/customization support",
    "lead time",
    "source link",
    "confidence_score",
    "scraped_at",
]

df = pd.DataFrame(records, columns=columns)

if len(df) < TARGET_ROWS:
    print(
        f"Only {len(df)} records were collected. "
        "Run again later, add more SEARCH_QUERIES, or increase max_results."
    )

df.head()


df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
try:
    df.to_excel(OUTPUT_XLSX, index=False)
except ModuleNotFoundError:
    print("Excel export skipped because openpyxl is not installed.")
    print("Install it in Jupyter with: !pip install openpyxl")

print(f"Saved CSV:  {OUTPUT_CSV}")
if Path(OUTPUT_XLSX).exists():
    print(f"Saved XLSX: {OUTPUT_XLSX}")

# %%
df


Saved CSV:  indian_manufacturers_100.csv
Saved XLSX: indian_manufacturers_100.xlsx


,name,website,"location (city, state, country)",categories served,MOQ range,production capacity,certifications,logistics support,packaging/customization support,lead time,source link,confidence_score,scraped_at
0,SFO Technologies,https://www.sfo.co.in,"Kochi, Kerala, India",Electronics / EMS,Not stated on source,Not stated on source,Not stated on source,Not stated on source,Not stated on source,Not stated on source,https://www.sfo.co.in/,0.00,2026-07-04 15:07:40
1,Dixon Technologies,https://www.dixoninfo.com,"Noida, Uttar Pradesh, India",Electronics / EMS,Not stated on source,Not stated on source,Not stated,Not stated on source,Mentions packaging/customization/private-label...,Not stated on source,https://www.dixoninfo.com/certifications,0.16,2026-07-04 15:07:41
2,Centum Electronics,https://www.centumelectronics.com,"Bengaluru, Karnataka, India",Electronics / EMS,Not stated on source,Not stated on source,Not stated on source,Not stated on source,Not stated on source,Not stated on source,https://www.centumelectronics.com/,0.00,2026-07-04 15:07:42
3,Kaynes Technology,https://www.kaynestechnology.co.in,"Mysuru, Karnataka, India",Electronics / EMS,Not stated on source,Not stated on source,Not stated,Not stated on source,Mentions packaging/customization/private-label...,Not stated on source,https://www.kaynestechnology.co.in,0.55,2026-07-04 15:07:44
4,Innova Captab,https://www.innovacaptab.com,"Panchkula, Haryana, India",Pharmaceuticals / CDMO,Not stated on source,Not stated on source,Not stated on source,Not stated on source,Not stated on source,Not stated on source,https://www.innovacaptab.com/,0.00,2026-07-04 15:07:44
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Featherlite,https://www.featherlitefurniture.com,"Bengaluru, Karnataka, India",Furniture,Not stated on source,Not stated on source,Not stated on source,Not stated on source,Not stated on source,Not stated on source,https://www.featherlitefurniture.com/,0.00,2026-07-04 15:08:32
96,Hindustan Syringes & Medical Devices,https://www.hmdhealthcare.com,"Faridabad, Haryana, India",Medical devices,Not stated on source,Not stated on source,Not stated on source,Not stated on source,Not stated on source,Not stated on source,https://www.hmdhealthcare.com/,0.00,2026-07-04 15:08:32
97,Trivitron Healthcare,https://www.trivitron.com,"Chennai, Tamil Nadu, India",Medical devices,Not stated on source,Not stated on source,Not stated,Not stated on source,Not stated on source,Not stated on source,https://www.trivitron.com/about-us,1.00,2026-07-04 15:08:33
98,Healthium Medtech,https://www.healthiummedtech.com,"Bengaluru, Karnataka, India",Medical devices,Not stated on source,Not stated on source,Not stated,Not stated on source,Not stated on source,Not stated on source,https://azapp-ci-hml-prod.azurewebsites.net/,0.23,2026-07-04 15:08:34
